# SmolVLM base: generated-token attention on 1,000 test questions
Enable Internet and one GPU or T4 ×2; attach `huyhuy12334/cache-smolvlm`.
**Cell 1 clones/pulls the entire GitHub repository** and activates the updated source.
Push changes to the configured branch before rerunning cell 1. Batch starts at **16 per GPU**;
only CUDA OOM reduces it. Progress bars cover validation, preparation and measurement.

Seed 42 selects 1,000 test questions without replacement. Greedy, at most 32 new tokens.
Query = generated content-token position, including self; EOS is excluded.
Mean heads → generated tokens within each answer → equal-weight nonempty questions.
Empty answers are reported, not replaced. Three curves show mean ±1 sample SD.
Histogram: 384 already-averaged visual-token values per layer; all 30 layers.

New run: run cells in order. **Redraw saved results: cell 1, then cells 6–9 only.**
Outputs remain in `/kaggle/working/generated_attention_outputs/`, outside the Git checkout.
Source changes affect measurement fingerprints; do not overwrite an existing run configuration
merely to update figures. Keep the original config/results for redraws.


In [ ]:
# 1. PULL ALL SOURCE from GitHub, install dependencies, and activate the updated code.
# Rerun this cell after pushing code; finish running workers before updating source.
import os
from pathlib import Path
SOURCE_REPOSITORY = "https://github.com/huyvanzzz/Log_Metrics.git"
SOURCE_BRANCH = "smolvlm-attention"  # Change only if your pushed code is on another branch.
SOURCE = Path('/kaggle/working/visualize-github-source')
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'

"""Standard-library-only Git setup, embedded verbatim in both Kaggle notebooks."""
import hashlib
import importlib
import importlib.util
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile


def ensure_no_workers(output_roots):
    for name in ('smolvlm_attention.runtime', 'smolvlm_attention.generated_runtime',
                 'dinov2_attention.runtime', 'smolvlm_probing.runtime',
                 'smolvlm_attention.sample_runtime', 'smolvlm_topk_benchmark.runtime',
                 'smolvlm_attention.combined_runtime'):
        module = sys.modules.get(name)
        if module is not None and any(process.poll() is None for process in getattr(module, '_PROCESSES', {}).values()):
            raise RuntimeError('A measurement worker is still running. Wait for it to finish before pulling source.')
    # The canonical probing output is shared by all notebooks. Also inspect it
    # after a kernel restart, when the new runtime is no longer in sys.modules.
    roots = set(map(Path, output_roots)) | {Path('/kaggle/working/smolvlm_probing_outputs'),
                                          Path('/kaggle/working/smolvlm_prompt_attention_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64_two_waves'),
                                          Path('/kaggle/working/smolvlm_combined_visual_attention_outputs')}
    for root in roots:
        for path in (Path(root) / 'logs').glob('*.pid.json'):
            pid = json.loads(path.read_text(encoding='utf-8'))['pid']
            try:
                os.kill(pid, 0)
            except (OSError, ProcessLookupError):
                continue
            raise RuntimeError(f'A measurement worker PID {pid} may still be running ({path}). Finish it before pulling.')


def sync_source(repository, branch, directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    def git(*args):
        return subprocess.check_output(['git', '-C', str(directory), *args], text=True).strip()
    if directory.exists():
        if not (directory / '.git').exists():
            raise RuntimeError(f'{directory} is not a Git checkout. Choose a new SOURCE directory.')
        if git('remote', 'get-url', 'origin').rstrip('/') != str(repository).rstrip('/'):
            raise RuntimeError('Existing source checkout has a different origin; choose a new SOURCE directory.')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Source checkout has local changes. Preserve them before pulling; no files were reset.')
        if git('branch', '--show-current') != branch:
            raise RuntimeError('Checkout branch differs from SOURCE_BRANCH. Use the matching branch or a new SOURCE directory.')
        subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', 'origin', branch], check=True)
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
                        str(repository), str(directory)], check=True)
    # Keep the historical source_archive_sha256 field reproducible without embedding stale source.
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for package in packages:
            folder = directory / package
            if not folder.is_dir():
                raise RuntimeError(f'Missing source package after pull: {package}')
            for path in sorted(folder.iterdir()):
                if path.suffix in ('.py', '.txt'):
                    entry = zipfile.ZipInfo(path.relative_to(directory).as_posix(), (2026, 1, 1, 0, 0, 0))
                    entry.compress_type = zipfile.ZIP_DEFLATED
                    archive.writestr(entry, path.read_bytes())
    return dict(commit=git('rev-parse', 'HEAD'), sha256=hashlib.sha256(buffer.getvalue()).hexdigest())


def activate_source(directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    for package in packages:
        for path in (directory / package).glob('*.py'):
            Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + '.')
               for package in packages):
            del sys.modules[name]
    sys.path[:] = [str(directory)] + [entry for entry in sys.path if entry != str(directory)]
    importlib.invalidate_caches()

ensure_no_workers([Path('/kaggle/working/attention_outputs'),
                   Path('/kaggle/working/generated_attention_outputs'),
                   Path('/kaggle/working/dinov2_attention_outputs')])
identity = sync_source(SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE)
SOURCE_COMMIT = identity["commit"]
SOURCE_ARCHIVE_SHA256 = identity["sha256"]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt')], check=True)
activate_source(SOURCE)
import torch, transformers, tokenizers
if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
    raise RuntimeError('Restart kernel after installing pinned packages, then run this setup cell again')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator')
OUTPUT = Path('/kaggle/working/generated_attention_outputs')
RUN_CONFIG = OUTPUT / 'generated_run.json'  # Existing results can be redrawn without rerunning preparation.
print({'source': str(SOURCE), 'commit': SOURCE_COMMIT, 'source_sha256': SOURCE_ARCHIVE_SHA256,
       'batch_per_gpu': 16, 'precision': 'float32',
       'gpus': [torch.cuda.get_device_name(i) for i in range(min(2, torch.cuda.device_count()))]})
print('Source updated. New run: continue cells in order. Redraw existing results: skip preparation/workers.')


In [ ]:
# 2. Download pinned prepared data and BASE checkpoint; use the existing Kaggle feature cache.
from smolvlm_attention.generated_runtime import resolve_generated_artifacts, prepare_generated_run
OUTPUT = Path('/kaggle/working/generated_attention_outputs')
CACHE_INPUT_DIR = ''  # Optional: explicit mounted cache root if automatic discovery fails.
CACHE_RECEIPT = None  # Optional: path to a previous validated_cache.json; inputs are rechecked.
artifacts = resolve_generated_artifacts(OUTPUT, CACHE_INPUT_DIR)
print({key: value for key, value in artifacts.items() if key != 'model'})
print(artifacts['model'])

In [ ]:
# 3. Validate cache ONCE (or reuse matching receipt), select 1,000 questions and prepare prompts.
# CPU/disk work: progress bars appear here; GPUs are used in the next cells.
RUN_CONFIG = prepare_generated_run(artifacts, source_archive_sha256=SOURCE_ARCHIVE_SHA256,
    source_commit=SOURCE_COMMIT, sample_count=1000, seed=42, max_new_tokens=32,
    receipt_path=CACHE_RECEIPT)
print(RUN_CONFIG)

In [ ]:
# 4. Launch base model on available GPUs: two 500-question shards or one 1,000-question shard.
from smolvlm_attention.generated_runtime import launch_generated_workers, wait_generated_workers
workers = launch_generated_workers(RUN_CONFIG)

In [ ]:
# 5. tqdm per GPU and total; batch starts at 16/GPU, reduced only on OOM.
# Interrupting this wait leaves workers running; rerun to reconnect.
wait_generated_workers(RUN_CONFIG)

In [ ]:
# 6. Verify full selected-ID coverage, combine samples equally and save numeric results/responses.
from smolvlm_attention.generated_runtime import load_generated_results
result, figure_metadata = load_generated_results(RUN_CONFIG)
FIGURES = OUTPUT / 'figures' / 'base'
from IPython.display import display, Image, FileLink
from smolvlm_attention.generated_plotting import (
    plot_attention_fractions, plot_generated_histogram, make_generated_zip)
print({key: result[key] for key in ('processed_count', 'valid_count', 'empty_count')})

In [ ]:
# 7. Figure 1: Image / Prompt text / Generated attention, mean ±1 SD across valid samples.
fraction_files = plot_attention_fractions(result['mean_groups'], result['std_groups'], FIGURES)
display(Image(filename=str(next(path for path in fraction_files if path.suffix == '.png'))))

In [ ]:
# 8. Figure 2: 384 averaged image-token attention values per layer, log-count histogram.
histogram_files = plot_generated_histogram(result['mean_visual'], FIGURES)
display(Image(filename=str(next(path for path in histogram_files if path.suffix == '.png'))))

In [ ]:
# 9. Download both figures as PNG/PDF, metadata and captions; no model/cache payload in ZIP.
zip_path = make_generated_zip(FIGURES, OUTPUT / 'generated_attention_figures.zip', figure_metadata)
display(FileLink(str(zip_path)))
print('Numeric summary:', OUTPUT / 'summary.npz')
print('Answers and token IDs:', OUTPUT / 'responses.jsonl')